# Import des bibliothèques nécessaires

In [14]:
from pathlib import Path
import pandas as pd
import re



# Chargement des données

In [15]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "01_paquets_phrases.csv", encoding="utf-8",)
df.head()

,nom_fichier,id_paquet,phrases_paquet
0,1884_12_La_joie_de_vivre._clean.txt,0,Comme six heures sonnaient au coucou de la sal...
1,1884_12_La_joie_de_vivre._clean.txt,1,"Il ajouta, après une hésitation: Tu devrais al..."
2,1884_12_La_joie_de_vivre._clean.txt,2,en voilà une morveuse qui peut se flatter de n...
3,1884_12_La_joie_de_vivre._clean.txt,3,Et on s’était à peine rencontré deux ou trois ...
4,1884_12_La_joie_de_vivre._clean.txt,4,Quelques gouttes de pluie volant dans l’ouraga...


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 653 entries, 0 to 652
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   nom_fichier     653 non-null    str  
 1   id_paquet       653 non-null    int64
 2   phrases_paquet  653 non-null    str  
dtypes: int64(1), str(2)
memory usage: 15.4 KB


# Préparation du dataframe 

In [17]:
df = df.rename(columns={
    "nom_fichier": "fichier",
    "id_paquet": "paquet_id",
    "phrases_paquet": "texte"
})

In [18]:
df["annee"] = df["fichier"].str.extract(r"^(\d{4})").astype(int)

In [19]:
ordre_romans= (
    df[["fichier", "annee"]]
    .drop_duplicates()
    .sort_values(["annee", "fichier"])
    .reset_index(drop=True)
)

ordre_romans["ordre_romans"] = range(1, len(ordre_romans) + 1)

df = df.merge(ordre_romans[["fichier", "ordre_romans"]], on="fichier", how="left")

In [20]:
df["roman"] = (
    df["fichier"]
    .str.replace(r"^\d{4}_\d+_", "", regex=True)
    .str.replace(r"_clean\.txt$", "", regex=True)
    .str.replace("_", " ")
)

In [21]:
df["nb_mots"] = df["texte"].str.split().str.len()

In [22]:
df = df[[
    "roman",
    "annee",
    "ordre_romans",
    "paquet_id",
    "texte",
    "nb_mots"
]]

In [23]:
df = df.sort_values(["ordre_romans", "paquet_id"]).reset_index(drop=True)

In [24]:
df["paquet_id"] = df.groupby("roman").cumcount() + 1

In [25]:
df.head()

,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,La joie de vivre.,1884,1,1,Comme six heures sonnaient au coucou de la sal...,265
1,La joie de vivre.,1884,1,2,"Il ajouta, après une hésitation: Tu devrais al...",156
2,La joie de vivre.,1884,1,3,en voilà une morveuse qui peut se flatter de n...,265
3,La joie de vivre.,1884,1,4,Et on s’était à peine rencontré deux ou trois ...,255
4,La joie de vivre.,1884,1,5,Quelques gouttes de pluie volant dans l’ouraga...,188


In [26]:
chemin_sortie = Path("..") /"data" /"2_processed" /"02_corpus_zola.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(chemin_sortie, index=False, encoding="utf-8")